# Article 1 v3 — análisis canónico

Definiciones, estado científico y siguientes pasos: [README](../README.md).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "article1" / "distillation.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from article1.analysis import (
    load_results, comparisons, summarize, routing_trajectories,
    support_summary, support_counts, plot_accuracy, plot_effect,
    plot_temperature, export,
)
OUT = ROOT / "OUTPUTS" / "article1_v3"
VERIFY_CACHES = False  # True requires sources/; CSV validation is always mandatory.
context = load_results(OUT)
print(context["validation"])
effects = comparisons(context)
tables, figures = {}, {}

In [ ]:
if VERIFY_CACHES:
    from article1.audit import audit
    from article1.rq2 import validate_cells
    cache_report = audit(OUT / "results.csv", source_root=OUT / "sources")
    if not cache_report["ok"]:
        raise RuntimeError(cache_report)
    display(cache_report)
    if context["temperature"] is not None:
        temperature_report = validate_cells(
            [OUT / "results.csv", OUT / "results_rq2_temperature.csv"],
            source_root=OUT / "sources", temperatures=(1, 4, 8),
            reject_foreign=OUT / "results_rq2_temperature.csv",
        )
        if not temperature_report["ok"] or temperature_report["counts"].get("valid_reusable") != 54:
            raise RuntimeError(temperature_report)
        display(temperature_report)

## RQ1 — Quién contribuye


In [ ]:
tables["rq1_paired"] = effects["routing"]
tables["rq1_summary"] = summarize(effects["routing"])
tables["rq1_trajectories"] = routing_trajectories(effects["routing"])
tables["rq1_oracle_gap"] = summarize(effects["oracle_gap"])
tables["rq1_controls"] = pd.concat([
    summarize(effects[name]).assign(control=name)
    for name in ("confidence_logit", "consensus_logit", "energy_logit")
], ignore_index=True)
for name in ("rq1_summary", "rq1_trajectories", "rq1_oracle_gap", "rq1_controls"):
    display(tables[name])
figures["rq1_student_accuracy"] = plot_accuracy(context["t8"])
figures["rq1_expertise_gain"] = plot_effect(effects["routing"], "EXPERT − FedDF (pp)")
display(figures["rq1_student_accuracy"], figures["rq1_expertise_gain"])
plt.close("all")

## RQ2-A — Logits frente a probabilidades


In [ ]:
tables["rq2_aggregation_paired"] = effects["aggregation"]
tables["rq2_aggregation_summary"] = summarize(effects["aggregation"])
tables["rq2_oracle_aggregation"] = summarize(effects["oracle_aggregation"])
display(tables["rq2_aggregation_summary"], tables["rq2_oracle_aggregation"])
figures["rq2_logit_probability_t8"] = plot_effect(effects["aggregation"], "EXPERT-prob − EXPERT-logit (pp)")
display(figures["rq2_logit_probability_t8"])
if "temperature" in effects:
    tables["rq2_temperature_paired"] = effects["temperature"]
    tables["rq2_temperature_summary"] = summarize(effects["temperature"], groups=["regime", "temperature"])
    display(tables["rq2_temperature_summary"])
    figures["rq2_temperature_sensitivity"] = plot_temperature(effects["temperature"])
    display(figures["rq2_temperature_sensitivity"])
else:
    print("Temperature sensitivity unavailable: results_rq2_temperature.csv is absent.")
plt.close("all")

## RQ2-B — Distribución completa frente a soporte restringido


In [ ]:
support = effects["support"]
tables["rq2_support_paired"] = support
tables["rq2_support_summary"] = support_summary(support)
display(support_counts(support), tables["rq2_support_summary"])
# Full-mask controls are kept visible even when their student effects are small.
control = support.merge(context["conditions"][["dataset", "regime", "seed", "M_density"]], on=["dataset", "regime", "seed"], validate="one_to_one")
tables["rq2_full_mask_control"] = control.loc[control.M_density.eq(1), ["dataset", "regime", "seed", "delta_accuracy_pp", "delta_target_nll", "delta_target_entropy"]]
display(tables["rq2_full_mask_control"])
figures["rq2_support_accuracy"] = plot_effect(support, "EXPERT-prob-SR − EXPERT-prob (pp)")
display(figures["rq2_support_accuracy"])
plt.close("all")

## Exportación de tablas y figuras


In [ ]:
export(OUT, tables, figures)
print(f"Exported {len(tables)} tables and {len(figures)} figures (PNG + PDF). Input CSVs unchanged.")
print("Supervised/proxy-size study: pending; see SIGUIENTES PASOS in README.")